# risk_budget_downside_vol_trendbreak_reentry_v1

Notebook-contained diagnostic experiment continuing the SOXL overfit / low-drawdown target loop.

## Governance

- Direction: `execution`
- Changed layer: execution / position sizing state machine
- Role: `diagnose` and `ablate`
- Canonical baseline: `risk_budget_usage_v1_actor_parity` as the retained low-drawdown risk-budget family.
- Direct comparison baseline: `risk_budget_downside_vol_target_v1_best_downside_vol`, because it was the prior near-frontier: MaxDD around 16.28% with +31pp excess.
- Target window: strict OOS full path plus fixed slices `test_2020`, `test_2022`, `test_2023_2024`.
- Acceptance criteria: strict-OOS `relative_return >= +30pp` and `max_dd <= 15%`, while not collapsing fixed slices.
- Changed mechanism: add absolute-momentum / trend-break risk-off state and explicit re-entry state to the prior downside-volatility path.
- Expected trade-off: absolute momentum should reduce long tail drawdown but may cut 2020 / 2023 bull participation.
- Failure mode: trend-break risk-off fires too broadly, reducing MaxDD slightly while destroying buy-hold-relative upside.
- Closest historical success: downside-vol target family brought MaxDD close to 15% while preserving +30pp excess.
- Closest historical failure: `recovery_reentry_2021_trendbreak_cap_v1` and broad trendbreak caps cut participation without solving the target.
- Why this is not duplicate: this tests trend-break/re-entry on the low-risk downside-vol path, plus an account-drawdown-conditioned variant, rather than applying broad trendbreak caps to a high-drawdown recovery branch.
- Comparison-table row key: `risk_budget_downside_vol_trendbreak_reentry_v1`.

## External References

- Antonacci, *Risk Premia Harvesting Through Dual Momentum*: absolute momentum is especially useful for volatility/drawdown control. https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2042750
- Bernardi/Bianchi/Bianco, *Smoothing volatility targeting*: smoothing can regularise leverage and turnover from realised-volatility targeting. https://arxiv.org/abs/2212.07288
- Grebenkov/Serror, *Following a Trend with an Exponential Moving Average*: EMA trend following has explicit risk/turnover trade-offs. https://arxiv.org/abs/1308.5658
- Qlib: keep the loop in the existing qlib-style research harness. https://github.com/microsoft/qlib


In [1]:
from __future__ import annotations

from itertools import product
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "runtime").exists():
            return candidate
    raise RuntimeError(f"could not locate repo root from {start}")

ROOT = find_repo_root()
CARD = ROOT / "runtime/execution/card/risk_budget_downside_vol_trendbreak_reentry_v1"
CARD.mkdir(parents=True, exist_ok=True)
EXPERIMENT = "risk_budget_downside_vol_trendbreak_reentry_v1"
BASELINE_PATH = ROOT / "runtime/execution/card/risk_budget_downside_vol_target_v1/risk_budget_downside_vol_target_v1_best_downside_vol_soxl_daily.csv"
RAW_PATH = ROOT / "runtime/model/card/risk_budget_usage_v1/risk_budget_usage_v1_actor_parity_soxl_daily.csv"

base = pd.read_csv(BASELINE_PATH)
base["datetime"] = pd.to_datetime(base["datetime"])
base = base.sort_values("datetime").reset_index(drop=True)
raw = pd.read_csv(RAW_PATH)
raw["datetime"] = pd.to_datetime(raw["datetime"])
raw = raw.sort_values("datetime").reset_index(drop=True)

WINDOWS = {
    "test_2020": ("2020-01-01", "2020-12-31"),
    "test_2022": ("2022-01-01", "2022-12-31"),
    "test_2023_2024": ("2023-01-01", "2024-12-31"),
}
print(BASELINE_PATH.relative_to(ROOT), base.shape, base["datetime"].min().date(), base["datetime"].max().date())

runtime/execution/card/risk_budget_downside_vol_target_v1/risk_budget_downside_vol_target_v1_best_downside_vol_soxl_daily.csv (1569, 11) 2020-01-02 2026-03-31


In [2]:
def metrics(frame: pd.DataFrame, *, start: str | None = None, end: str | None = None) -> dict:
    z = frame.copy()
    if start is not None:
        z = z[z["datetime"] >= pd.Timestamp(start)]
    if end is not None:
        z = z[z["datetime"] <= pd.Timestamp(end)]
    prices = z["price"].to_numpy(dtype=float)
    positions = z["position"].to_numpy(dtype=float)
    returns = np.diff(prices) / prices[:-1]
    strategy_returns = positions[:-1] * returns
    nav = np.concatenate([[1.0], np.cumprod(1.0 + strategy_returns)])
    buyhold = np.concatenate([[1.0], np.cumprod(1.0 + returns)])
    peak = np.maximum.accumulate(nav)
    drawdown = (nav - peak) / peak
    stdev = float(np.std(strategy_returns))
    return {
        "rows": int(len(z)),
        "start": z["datetime"].iloc[0].date().isoformat(),
        "end": z["datetime"].iloc[-1].date().isoformat(),
        "return": float(nav[-1] - 1.0) * 100.0,
        "sharpe": float(np.mean(strategy_returns) / stdev * np.sqrt(252)) if stdev > 0 else 0.0,
        "max_dd": abs(float(np.min(drawdown))) * 100.0,
        "buy_and_hold_return": float(buyhold[-1] - 1.0) * 100.0,
        "relative_return": float(nav[-1] - buyhold[-1]) * 100.0,
        "avg_pos": float(np.mean(positions)),
        "trades": int(np.sum(np.abs(np.diff(positions)) > 0.001)),
    }


def row_for(frame: pd.DataFrame, label: str) -> dict:
    row = {"label": label, **metrics(frame)}
    for name, (start, end) in WINDOWS.items():
        m = metrics(frame, start=start, end=end)
        for key in ["return", "sharpe", "max_dd", "relative_return", "avg_pos", "trades"]:
            row[f"{name}_{key}"] = m[key]
    row["target_pass"] = bool(row["relative_return"] >= 30.0 and row["max_dd"] <= 15.0)
    row["target_gap"] = max(0.0, row["max_dd"] - 15.0) * 4.0 + max(0.0, 30.0 - row["relative_return"])
    return row


def attach_nav(frame: pd.DataFrame) -> pd.DataFrame:
    z = frame.copy()
    prices = z["price"].to_numpy(dtype=float)
    positions = z["position"].to_numpy(dtype=float)
    returns = np.diff(prices) / prices[:-1]
    strategy_nav = np.concatenate([[1.0], np.cumprod(1.0 + positions[:-1] * returns)])
    buyhold_nav = np.concatenate([[1.0], np.cumprod(1.0 + returns)])
    z["strategy_nav"] = strategy_nav * 1_000_000.0
    z["buyhold_nav"] = buyhold_nav * 1_000_000.0
    z["excess"] = z["strategy_nav"] - z["buyhold_nav"]
    z["excess_pct"] = z["excess"] / z["buyhold_nav"] * 100.0
    actions = ["HOLD"]
    for delta in np.diff(positions):
        actions.append("BUY" if delta > 0.001 else "SELL" if delta < -0.001 else "HOLD")
    z["action"] = actions
    leading = ["datetime", "strategy_nav", "price", "buyhold_nav", "position", "excess", "excess_pct", "action"]
    return z[leading + [c for c in z.columns if c not in leading]]

raw_row = row_for(raw[["datetime", "price", "position"]], "risk_budget_usage_v1_actor_parity")
base_row = row_for(base[["datetime", "price", "position"]], "risk_budget_downside_vol_target_v1_best_downside_vol")
display(pd.DataFrame([raw_row, base_row])[["label", "return", "sharpe", "max_dd", "relative_return", "avg_pos", "trades", "target_pass"]])

,label,return,sharpe,max_dd,relative_return,avg_pos,trades,target_pass
0,risk_budget_usage_v1_actor_parity,233.856985,1.028952,21.375029,87.710263,0.180393,1320,False
1,risk_budget_downside_vol_target_v1_best_downsi...,177.479796,0.960142,16.277862,31.333073,0.163345,401,False


In [3]:
def max_drawdown_span(frame: pd.DataFrame) -> dict:
    prices = frame["price"].to_numpy(dtype=float)
    positions = frame["position"].to_numpy(dtype=float)
    returns = np.diff(prices) / prices[:-1]
    nav = np.concatenate([[1.0], np.cumprod(1.0 + positions[:-1] * returns)])
    peak = np.maximum.accumulate(nav)
    drawdown = (nav - peak) / peak
    trough_idx = int(np.argmin(drawdown))
    peak_idx = int(np.argmax(nav[: trough_idx + 1]))
    return {
        "peak_date": frame["datetime"].iloc[peak_idx].date().isoformat(),
        "trough_date": frame["datetime"].iloc[trough_idx].date().isoformat(),
        "max_dd": abs(float(drawdown[trough_idx])) * 100.0,
        "peak_position": float(frame["position"].iloc[peak_idx]),
        "trough_position": float(frame["position"].iloc[trough_idx]),
        "peak_price": float(frame["price"].iloc[peak_idx]),
        "trough_price": float(frame["price"].iloc[trough_idx]),
    }

drawdown_span = max_drawdown_span(base)
display(Markdown("## Downside-Vol Drawdown Span"))
display(pd.Series(drawdown_span).to_frame("value"))

## Downside-Vol Drawdown Span

,value
peak_date,2024-07-10
trough_date,2025-04-08
max_dd,16.277862
peak_position,0.23218
trough_position,0.045436
peak_price,68.589996
trough_price,8.25


In [4]:
price = base["price"].to_numpy(dtype=float)
desired = base["position"].to_numpy(dtype=float)
ma_cache = {w: base["price"].rolling(w, min_periods=max(10, w // 2)).mean().to_numpy(dtype=float) for w in [20, 50, 100, 200]}
mom_cache = {w: (base["price"] / base["price"].shift(w) - 1.0).to_numpy(dtype=float) for w in [10, 21, 42]}
high_drawdown_63 = (base["price"] / base["price"].rolling(63, min_periods=20).max() - 1.0).to_numpy(dtype=float)


def trendbreak_path(
    *,
    dd_trigger: float | None,
    long_ma: int,
    short_ma: int,
    momentum_window: int,
    exit_momentum_window: int,
    break_momentum: float,
    break_drawdown: float,
    reentry_momentum: float,
    reentry_drawdown: float,
    off_cap: float,
    trade_band: float,
) -> tuple[pd.DataFrame, dict]:
    state = False
    current = 0.0
    nav = 1.0
    peak = 1.0
    previous_price: float | None = None
    positions: list[float] = []
    states: list[bool] = []
    account_drawdowns: list[float] = []
    entries = 0
    exits = 0
    off_days = 0
    long_ma_values = ma_cache[long_ma]
    short_ma_values = ma_cache[short_ma]
    momentum_values = mom_cache[momentum_window]
    exit_momentum_values = mom_cache[exit_momentum_window]
    for idx, px in enumerate(price):
        if previous_price is not None and previous_price > 0.0:
            nav *= 1.0 + current * (px / previous_price - 1.0)
        peak = max(peak, nav)
        account_dd = max(1.0 - nav / max(peak, 1e-12), 0.0)
        dd_ready = True if dd_trigger is None else account_dd >= float(dd_trigger)
        if not state and dd_ready and px < long_ma_values[idx] and momentum_values[idx] <= break_momentum and high_drawdown_63[idx] <= -break_drawdown:
            state = True
            entries += 1
        elif state and px > short_ma_values[idx] and exit_momentum_values[idx] >= reentry_momentum and high_drawdown_63[idx] >= -reentry_drawdown:
            state = False
            exits += 1
        target = min(desired[idx], off_cap) if state else desired[idx]
        if state:
            off_days += 1
        threshold = 0.0 if state and target < current else trade_band
        if abs(target - current) >= threshold:
            current = float(target)
        positions.append(current)
        states.append(state)
        account_drawdowns.append(account_dd)
        previous_price = px
    out = base.copy()
    out["pre_trendbreak_position"] = out["position"]
    out["position"] = positions
    out["trendbreak_risk_off"] = states
    out["strategy_account_drawdown"] = account_drawdowns
    meta = {"entries": entries, "exits": exits, "off_days": off_days}
    return out, meta


def screen_rows(account_dd_conditioned: bool) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    rows = []
    candidates: dict[str, pd.DataFrame] = {}
    dd_triggers = [0.07, 0.09, 0.11] if account_dd_conditioned else [None]
    for dd_trigger, long_ma, short_ma, momentum_window, exit_momentum_window, break_momentum, break_drawdown, reentry_momentum, reentry_drawdown, off_cap, trade_band in product(
        dd_triggers,
        [100, 200],
        [20, 50],
        [21, 42],
        [10, 21],
        [0.0, 0.03],
        [0.08, 0.12],
        [0.0, 0.03],
        [0.12, 0.20],
        [0.02, 0.05, 0.10],
        [0.02, 0.04],
    ):
        if short_ma > long_ma:
            continue
        candidate, meta = trendbreak_path(
            dd_trigger=dd_trigger,
            long_ma=long_ma,
            short_ma=short_ma,
            momentum_window=momentum_window,
            exit_momentum_window=exit_momentum_window,
            break_momentum=break_momentum,
            break_drawdown=break_drawdown,
            reentry_momentum=reentry_momentum,
            reentry_drawdown=reentry_drawdown,
            off_cap=off_cap,
            trade_band=trade_band,
        )
        label = (
            f"{'acctdd' if account_dd_conditioned else 'price'}_dd{dd_trigger if dd_trigger is not None else 'none'}"
            f"_ma{long_ma}_{short_ma}_mom{momentum_window}_{exit_momentum_window}"
            f"_bm{break_momentum:.2f}_bd{break_drawdown:.2f}_rm{reentry_momentum:.2f}_rd{reentry_drawdown:.2f}"
            f"_cap{off_cap:.2f}_band{trade_band:.2f}"
        )
        row = row_for(candidate, label)
        row.update({
            "account_dd_conditioned": account_dd_conditioned,
            "dd_trigger": dd_trigger,
            "long_ma": long_ma,
            "short_ma": short_ma,
            "momentum_window": momentum_window,
            "exit_momentum_window": exit_momentum_window,
            "break_momentum": break_momentum,
            "break_drawdown": break_drawdown,
            "reentry_momentum": reentry_momentum,
            "reentry_drawdown": reentry_drawdown,
            "off_cap": off_cap,
            "trade_band": trade_band,
            **meta,
        })
        rows.append(row)
        candidates[label] = candidate
    screen = pd.DataFrame(rows).sort_values(["target_pass", "target_gap", "max_dd", "relative_return"], ascending=[False, True, True, False])
    return screen, candidates

price_screen, price_candidates = screen_rows(False)
account_screen, account_candidates = screen_rows(True)
price_screen.to_csv(CARD / "price_trendbreak_reentry_screen.csv", index=False)
account_screen.to_csv(CARD / "account_dd_trendbreak_reentry_screen.csv", index=False)

print("price screen", len(price_screen), "pass", int(price_screen["target_pass"].sum()))
print("account-dd screen", len(account_screen), "pass", int(account_screen["target_pass"].sum()))
display(price_screen.head(10)[["label", "return", "sharpe", "max_dd", "relative_return", "avg_pos", "trades", "off_days", "entries", "exits", "target_pass", "target_gap"]])
display(account_screen.head(10)[["label", "return", "sharpe", "max_dd", "relative_return", "avg_pos", "trades", "off_days", "entries", "exits", "target_pass", "target_gap"]])

price screen 1536 pass 0
account-dd screen 4608 pass 0


,label,return,sharpe,max_dd,relative_return,avg_pos,trades,off_days,entries,exits,target_pass,target_gap
394,price_ddnone_ma100_50_mom21_10_bm0.00_bd0.08_r...,149.509681,0.910398,15.489142,3.362959,0.149818,359,647,19,18,False,28.593610
418,price_ddnone_ma100_50_mom21_10_bm0.00_bd0.12_r...,149.509681,0.910398,15.489142,3.362959,0.149818,359,647,19,18,False,28.593610
490,price_ddnone_ma100_50_mom21_21_bm0.00_bd0.08_r...,149.260223,0.909699,15.489142,3.113500,0.149346,357,666,18,17,False,28.843069
514,price_ddnone_ma100_50_mom21_21_bm0.00_bd0.12_r...,149.260223,0.909699,15.489142,3.113500,0.149346,357,666,18,17,False,28.843069
442,price_ddnone_ma100_50_mom21_10_bm0.03_bd0.08_r...,148.901437,0.908234,15.489142,2.754714,0.149756,359,650,19,18,False,29.201855
466,price_ddnone_ma100_50_mom21_10_bm0.03_bd0.12_r...,148.901437,0.908234,15.489142,2.754714,0.149756,359,650,19,18,False,29.201855
538,price_ddnone_ma100_50_mom21_21_bm0.03_bd0.08_r...,148.652587,0.907534,15.489142,2.505864,0.149284,357,669,18,17,False,29.450705
562,price_ddnone_ma100_50_mom21_21_bm0.03_bd0.12_r...,148.652587,0.907534,15.489142,2.505864,0.149284,357,669,18,17,False,29.450705
106,price_ddnone_ma100_20_mom21_21_bm0.00_bd0.08_r...,148.130844,0.905801,15.489142,1.984122,0.149314,357,663,18,17,False,29.972447
130,price_ddnone_ma100_20_mom21_21_bm0.00_bd0.12_r...,148.130844,0.905801,15.489142,1.984122,0.149314,357,663,18,17,False,29.972447


,label,return,sharpe,max_dd,relative_return,avg_pos,trades,off_days,entries,exits,target_pass,target_gap
1547,acctdd_dd0.09_ma100_20_mom21_10_bm0.00_bd0.08_...,159.277092,0.927840,15.406526,13.130369,0.156982,222,163,5,5,False,18.495734
1571,acctdd_dd0.09_ma100_20_mom21_10_bm0.00_bd0.12_...,159.277092,0.927840,15.406526,13.130369,0.156982,222,163,5,5,False,18.495734
1595,acctdd_dd0.09_ma100_20_mom21_10_bm0.03_bd0.08_...,159.277092,0.927840,15.406526,13.130369,0.156982,222,163,5,5,False,18.495734
1619,acctdd_dd0.09_ma100_20_mom21_10_bm0.03_bd0.12_...,159.277092,0.927840,15.406526,13.130369,0.156982,222,163,5,5,False,18.495734
1931,acctdd_dd0.09_ma100_50_mom21_10_bm0.00_bd0.08_...,159.277092,0.927840,15.406526,13.130369,0.156982,222,163,5,5,False,18.495734
1955,acctdd_dd0.09_ma100_50_mom21_10_bm0.00_bd0.12_...,159.277092,0.927840,15.406526,13.130369,0.156982,222,163,5,5,False,18.495734
1979,acctdd_dd0.09_ma100_50_mom21_10_bm0.03_bd0.08_...,159.277092,0.927840,15.406526,13.130369,0.156982,222,163,5,5,False,18.495734
2003,acctdd_dd0.09_ma100_50_mom21_10_bm0.03_bd0.12_...,159.277092,0.927840,15.406526,13.130369,0.156982,222,163,5,5,False,18.495734
1643,acctdd_dd0.09_ma100_20_mom21_21_bm0.00_bd0.08_...,158.443672,0.925146,15.406526,12.296949,0.156827,221,166,5,5,False,19.329154
1667,acctdd_dd0.09_ma100_20_mom21_21_bm0.00_bd0.12_...,158.443672,0.925146,15.406526,12.296949,0.156827,221,166,5,5,False,19.329154


In [5]:
# Select the best diagnostic candidate by explicit target gap, but keep baseline in the summary.
combined_screen = pd.concat([price_screen.assign(screen="price"), account_screen.assign(screen="account_dd")], ignore_index=True)
combined_screen = combined_screen.sort_values(["target_pass", "target_gap", "max_dd", "relative_return"], ascending=[False, True, True, False])
selected = combined_screen.iloc[0].to_dict()
selected_label = selected["label"]
selected_frame = price_candidates.get(selected_label) or account_candidates[selected_label]
selected_row = row_for(selected_frame, f"{EXPERIMENT}_selected")
selected_row.update({"source_label": selected_label, "source_screen": selected["screen"]})

summary = pd.DataFrame([raw_row, base_row, selected_row])
summary.to_csv(CARD / "selection_summary.csv", index=False)
attach_nav(selected_frame).to_csv(CARD / f"{EXPERIMENT}_selected_soxl_daily.csv", index=False)

selection_payload = {
    "experiment": EXPERIMENT,
    "baseline_path": str(BASELINE_PATH.relative_to(ROOT)),
    "raw_baseline_path": str(RAW_PATH.relative_to(ROOT)),
    "drawdown_span": drawdown_span,
    "price_screen_rows": int(len(price_screen)),
    "price_target_pass_rows": int(price_screen["target_pass"].sum()),
    "account_dd_screen_rows": int(len(account_screen)),
    "account_dd_target_pass_rows": int(account_screen["target_pass"].sum()),
    "selected_label": selected_label,
    "selected_screen": selected["screen"],
    "selected": selected_row,
    "decision": "diagnostic_keep_no_promotion",
}
(CARD / "selection.json").write_text(json.dumps(selection_payload, indent=2), encoding="utf-8")

display(summary[["label", "return", "sharpe", "max_dd", "relative_return", "test_2020_return", "test_2020_max_dd", "test_2022_return", "test_2022_max_dd", "test_2023_2024_return", "test_2023_2024_max_dd", "avg_pos", "trades", "target_pass"]])

/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_71650/3972143540.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_screen = pd.concat([price_screen.assign(screen="price"), account_screen.assign(screen="account_dd")], ignore_index=True)


,label,return,sharpe,max_dd,relative_return,test_2020_return,test_2020_max_dd,test_2022_return,test_2022_max_dd,test_2023_2024_return,test_2023_2024_max_dd,avg_pos,trades,target_pass
0,risk_budget_usage_v1_actor_parity,233.856985,1.028952,21.375029,87.710263,38.452544,17.626678,-8.872797,18.869923,35.750847,19.462175,0.180393,1320,False
1,risk_budget_downside_vol_target_v1_best_downsi...,177.479796,0.960142,16.277862,31.333073,19.285428,15.489142,-7.628252,11.189810,32.578923,15.994587,0.163345,401,False
2,risk_budget_downside_vol_trendbreak_reentry_v1...,159.277092,0.927840,15.406526,13.130369,21.885141,15.406526,-8.971336,10.569515,31.871707,13.445546,0.156982,222,False


In [6]:
readout = f"""
## Final Readout

- Prior near-frontier baseline `{base_row['label']}`: return `{base_row['return']:.2f}%`, Sharpe `{base_row['sharpe']:.3f}`, MaxDD `{base_row['max_dd']:.2f}%`, relative return `{base_row['relative_return']:.2f}pp`.
- Baseline max drawdown span: `{drawdown_span['peak_date']}` to `{drawdown_span['trough_date']}`; position was already only `{drawdown_span['trough_position']:.3f}` at trough.
- Price-only trendbreak/re-entry screen rows: `{len(price_screen)}`; target-pass rows: `{int(price_screen['target_pass'].sum())}`.
- Account-DD-conditioned trendbreak/re-entry screen rows: `{len(account_screen)}`; target-pass rows: `{int(account_screen['target_pass'].sum())}`.
- Selected diagnostic candidate `{selected_label}`: return `{selected_row['return']:.2f}%`, MaxDD `{selected_row['max_dd']:.2f}%`, relative `{selected_row['relative_return']:.2f}pp`, target pass `{selected_row['target_pass']}`.

Decision: diagnostic keep / no runtime promotion. The trendbreak state machine can lower path drawdown modestly, but every variant that gets close to the 15% drawdown line gives up too much buy-hold-relative return. The root cause is that the downside-vol path is already very low exposure at the April 2025 trough; further drawdown reduction requires exiting earlier in the 2024-2025 decline, and those earlier exits also cut major bull/rebound participation. The next experiment should move away from binary trendbreak caps and test a more selective crash-risk predictor or retrain/risk-budget objective that learns earlier transition timing directly, with walk-forward actor validation.
"""
display(Markdown(readout))


## Final Readout

- Prior near-frontier baseline `risk_budget_downside_vol_target_v1_best_downside_vol`: return `177.48%`, Sharpe `0.960`, MaxDD `16.28%`, relative return `31.33pp`.
- Baseline max drawdown span: `2024-07-10` to `2025-04-08`; position was already only `0.045` at trough.
- Price-only trendbreak/re-entry screen rows: `1536`; target-pass rows: `0`.
- Account-DD-conditioned trendbreak/re-entry screen rows: `4608`; target-pass rows: `0`.
- Selected diagnostic candidate `acctdd_dd0.09_ma100_20_mom21_10_bm0.00_bd0.08_rm0.00_rd0.20_cap0.10_band0.04`: return `159.28%`, MaxDD `15.41%`, relative `13.13pp`, target pass `False`.

Decision: diagnostic keep / no runtime promotion. The trendbreak state machine can lower path drawdown modestly, but every variant that gets close to the 15% drawdown line gives up too much buy-hold-relative return. The root cause is that the downside-vol path is already very low exposure at the April 2025 trough; further drawdown reduction requires exiting earlier in the 2024-2025 decline, and those earlier exits also cut major bull/rebound participation. The next experiment should move away from binary trendbreak caps and test a more selective crash-risk predictor or retrain/risk-budget objective that learns earlier transition timing directly, with walk-forward actor validation.
